In [3]:
from collections import Counter
from itertools import permutations
from math import log, exp, inf
from pprint import pprint

# ============================================================
# DATA AND SHARED HELPERS
# ============================================================

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]


def show_table(title, rows, headers):
    print(f"\n{title}")
    print(" | ".join(headers))
    print("-" * 65)
    for row in rows:
        print(" | ".join(str(x) for x in row))


def fmt(x):
    if x == inf:
        return "inf"
    if x == -inf:
        return "-inf"
    if isinstance(x, float):
        return f"{x:.4f}"
    return str(x)


# Task 1: Tokenization
1.	Write tokenize(sentence) that lowercases, splits on spaces and adds <s> and </s>.
2.	Apply it to every training and test sentence and print the results.
3.	Print the total number of tokens in the training corpus with markers and without markers.
4.	Write build_vocab(sentences) and print V and the sorted vocabulary.


In [9]:
train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

# 1. Tokenize sentences
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

# 2. Apply tokenization to training and test sentences
train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("Training tokens:")
print(train_tokens)

print("\nTest tokens:")
print(test_tokens)

# 3. Count training tokens with and without markers
with_markers = sum(len(s) for s in train_tokens)
without_markers = sum(len(s) - 2 for s in train_tokens)

print("\nTokens with markers:", with_markers)
print("Tokens without markers:", without_markers)

# 4. Build vocabulary (exclude <s>, include </s>)
def build_vocab(sentences):
    return {
        word
        for sentence in sentences
        for word in tokenize(sentence)
        if word != "<s>"
    }

V = build_vocab(train)

print("\nVocabulary V:", V)
print("Sorted vocabulary:", sorted(V))
print("Vocabulary size:", len(V))

Training tokens:
[['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>'], ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>'], ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>'], ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']]

Test tokens:
[['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>'], ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']]

Tokens with markers: 30
Tokens without markers: 22

Vocabulary V: {'ate', 'rug', 'bone', 'sat', 'dog', '</s>', 'cat', 'fish', 'the', 'on', 'mat'}
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']
Vocabulary size: 11


# Task 2: Generating N-grams
1.	Write get_ngrams(tokens, n) that returns a list of tuples.
2.	Print the unigrams, bigrams and trigrams of the first training sentence.
3.	For every sentence, verify with code that m tokens give m − n + 1 N-grams for n = 1, 2 and 3.
4.	Show what get_ngrams returns when n is larger than the sentence length, and make sure it does not crash.


In [10]:
def get_ngrams(tokens, n):
    if n <= 0:
        raise ValueError("n must be positive")

    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]

# 2. Print n-grams of the first training sentence
first = train_tokens[0]

print("Unigrams:", get_ngrams(first, 1))
print("Bigrams:", get_ngrams(first, 2))
print("Trigrams:", get_ngrams(first, 3))

# 3. Verify the number of n-grams in every sentence
print("\nVerify n-gram counts:")

for i, tokens in enumerate(train_tokens, start=1):
    m = len(tokens)

    for n in (1, 2, 3):
        actual = len(get_ngrams(tokens, n))
        expected = max(0, m - n + 1)

        assert actual == expected

        print(
            f"Sentence {i}, n={n}: "
            f"m={m}, actual={actual}, expected={expected}"
        )

# 4. Test when n exceeds sentence length
print("\nn larger than sentence length:")
print(get_ngrams(first, len(first) + 1))

Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Verify n-gram counts:
Sentence 1, n=1: m=8, actual=8, expected=8
Sentence 1, n=2: m=8, actual=7, expected=7
Sentence 1, n=3: m=8, actual=6, expected=6
Sentence 2, n=1: m=7, actual=7, expected=7
Sentence 2, n=2: m=7, actual=6, expected=6
Sentence 2, n=3: m=7, actual=5, expected=5
Sentence 3, n=1: m=8, actual=8, expected=8
Sentence 3, n=2: m=8, actual=7, expected=7
Sentence 3, n=3: m=8, actual=6, expected=6
Sentence 4, n=1: m=7, actual=7, expected=7
Sentence 4, n=2: m=7, actual=6, expected=6
Sentence 4, n=3: m=7, actual=5, expected=5

n larger than sentence length:
[]


# Task 3: Counting N-grams
1.	Use collections.Counter to build unigram, bigram and trigram counts for the training corpus.
2.	Print each count table sorted from highest to lowest count.
3.	Print the 5 most common bigrams and the number of distinct unigrams, bigrams and trigrams.
4.	Write count_of(ngram) that returns 0 for an unseen N-gram instead of raising an error. Test it on one seen and one unseen bigram.


In [11]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))

print("\nTASK 3: N-GRAM COUNTS")


def print_counts(title, counter):
    print(f"\n{title}")
    for ngram, count in sorted(
        counter.items(), key=lambda item: (-item[1], item[0])
    ):
        print(f"{ngram}: {count}")


print_counts("Unigram counts", unigram_counts)
print_counts("Bigram counts", bigram_counts)
print_counts("Trigram counts", trigram_counts)

print("\nFive most common bigrams:")
print(bigram_counts.most_common(5))

distinct_counts = {
    "Unigrams": len(unigram_counts),
    "Bigrams": len(bigram_counts),
    "Trigrams": len(trigram_counts)
}

show_table(
    "Distinct N-gram counts",
    list(distinct_counts.items()),
    ["N-gram type", "Distinct count"]
)


def count_of(ngram):
    """Return zero for an unseen N-gram."""
    n = len(ngram)
    if n == 1:
        return unigram_counts[tuple(ngram)]
    if n == 2:
        return bigram_counts[tuple(ngram)]
    if n == 3:
        return trigram_counts[tuple(ngram)]
    raise ValueError("Only unigrams, bigrams, and trigrams are supported")


print("\nSeen bigram ('the', 'cat'):", count_of(("the", "cat")))
print("Unseen bigram ('cat', 'bone'):", count_of(("cat", "bone")))


TASK 3: N-GRAM COUNTS

Unigram counts
('the',): 8
('</s>',): 4
('<s>',): 4
('ate',): 2
('cat',): 2
('dog',): 2
('on',): 2
('sat',): 2
('bone',): 1
('fish',): 1
('mat',): 1
('rug',): 1

Bigram counts
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1

Trigram counts
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', '</s>'): 1
('the', 'mat', '</s>'): 1
('

# Task 4: Unigram model
1.	Write unigram_prob(w) using MLE, P(w) = C(w) / N.
2.	Print all unigram probabilities as a table and assert that they sum to 1.
3.	Write unigram_sentence_prob(sentence) and print the result for both test sentences.
4.	Use itertools.permutations on the words of the first test sentence and show that different word orders can give the same unigram probability.


In [27]:
from collections import Counter
from itertools import permutations

# Rebuild unigram counts
unigram_counts = Counter()

for sentence in train:
    tokens = tokenize(sentence)
    unigram_counts.update((w,) for w in tokens)

# Vocabulary excludes <s> but includes </s>
Vocab = {
    w
    for sentence in train
    for w in tokenize(sentence)
    if w != "<s>"
}

# Count tokens belonging to the vocabulary only
N = sum(unigram_counts[(w,)] for w in Vocab)

# 1. Unigram MLE probability
def unigram_prob(w):
    return unigram_counts[(w,)] / N

# 2. Unigram sentence probability
def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1.0

    for w in tokens[1:]:
        p *= unigram_prob(w)

    return p

# Print unigram probabilities
print("\nUnigram MLE probabilities")
print(f"{'Word':<12} {'Count':<8} {'P(word)':<10}")
print("-" * 30)

for word in sorted(Vocab):
    print(
        f"{word:<12} "
        f"{unigram_counts[(word,)]:<8} "
        f"{unigram_prob(word):.4f}"
    )

# Assert probabilities sum to 1
total = sum(unigram_prob(w) for w in Vocab)
print("\nSum of unigram probabilities:", round(total, 4))

assert abs(total - 1.0) < 1e-12
print("Unigram probabilities sum to 1.")

# 3. Test sentence probabilities
print("\nUnigram sentence probabilities:")
for sentence in test:
    print(repr(sentence), f"{unigram_sentence_prob(sentence):.8f}")

# 4. Permutations of first test sentence
print("\nPermutations of the first test sentence:")
words = test[0].split()
perms = list(dict.fromkeys(permutations(words)))

for perm in perms[:6]:
    sentence = " ".join(perm)
    print(
        f"{sentence:<35} "
        f"{unigram_sentence_prob(sentence):.8f}"
    )

print("\nAll permutations have the same unigram probability.")


Unigram MLE probabilities
Word         Count    P(word)   
------------------------------
</s>         4        0.1538
ate          2        0.0769
bone         1        0.0385
cat          2        0.0769
dog          2        0.0769
fish         1        0.0385
mat          1        0.0385
on           2        0.0769
rug          1        0.0385
sat          2        0.0769
the          8        0.3077

Sum of unigram probabilities: 1.0
Unigram probabilities sum to 1.

Unigram sentence probabilities:
'the cat sat on the rug' 0.00000025
'the fish sat on the mat' 0.00000013

Permutations of the first test sentence:
the cat sat on the rug              0.00000025
the cat sat on rug the              0.00000025
the cat sat the on rug              0.00000025
the cat sat the rug on              0.00000025
the cat sat rug on the              0.00000025
the cat sat rug the on              0.00000025

All permutations have the same unigram probability.


# Task 5: Bigram model (MLE)
1.	Write bigram_prob(prev, w) using P(w | v) = C(v, w) / C(v).
2.	Write next_word_distribution(prev) that returns a dictionary of next words and probabilities. Print it for “the” and assert that it sums to 1.
3.	Write bigram_sentence_prob(sentence) using the chain rule and print the result for both test sentences.
4.	Write predict_next(prev) that returns the most probable next word (break ties alphabetically) and test it on at least five different words.


In [19]:
def bigram_prob(prev, w):
    denominator = unigram_counts[(prev,)]
    if denominator == 0:
        return 0.0
    return bigram_counts[(prev, w)] / denominator


def next_word_distribution(prev):
    denominator = unigram_counts[(prev,)]
    if denominator == 0:
        return {}

    return {
        w: bigram_counts[(prev, w)] / denominator
        for w in sorted(Vocab)
        if bigram_counts[(prev, w)] > 0
    }


print("\nTASK 5: BIGRAM MODEL")
print('Next-word distribution for "the":')
the_distribution = next_word_distribution("the")
pprint(the_distribution)
assert abs(sum(the_distribution.values()) - 1.0) < 1e-12
print("Distribution sums to 1.")


def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1.0
    for i in range(1, len(tokens)):
        p *= bigram_prob(tokens[i - 1], tokens[i])
    return p


print("\nBigram MLE sentence probabilities:")
for s in test:
    print(repr(s), fmt(bigram_sentence_prob(s)))


def predict_next(prev):
    distribution = next_word_distribution(prev)
    if not distribution:
        return None
    return min(
        distribution,
        key=lambda w: (-distribution[w], w)
    )


print("\nNext-word predictions:")
for word in ["the", "cat", "dog", "sat", "ate", "on"]:
    print(f"{word!r} -> {predict_next(word)!r}")



TASK 5: BIGRAM MODEL
Next-word distribution for "the":
{'bone': 0.125,
 'cat': 0.25,
 'dog': 0.25,
 'fish': 0.125,
 'mat': 0.125,
 'rug': 0.125}
Distribution sums to 1.

Bigram MLE sentence probabilities:
'the cat sat on the rug' 0.0156
'the fish sat on the mat' 0.0000

Next-word predictions:
'the' -> 'cat'
'cat' -> 'ate'
'dog' -> 'ate'
'sat' -> 'on'
'ate' -> 'the'
'on' -> 'the'


# Task 6: Trigram model
1.	Write trigram_prob(w1, w2, w3) using P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2).
2.	Write trigram_sentence_prob(sentence). Use the bigram model for the first word given <s> and the trigram model for all other words. Print the result for both test sentences.
3.	Print the number of distinct unigrams, bigrams and trigrams in a small table.
4.	For each test sentence, list its unseen bigrams and its unseen trigrams, then print which model has more unseen N-grams.


In [20]:
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts[(w1, w2)]
    if denominator == 0:
        return 0.0
    return trigram_counts[(w1, w2, w3)] / denominator


def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    if len(tokens) < 2:
        return 1.0

    # First predicted word uses the bigram model.
    p = bigram_prob(tokens[0], tokens[1])

    # Remaining words use the trigram model.
    for i in range(2, len(tokens)):
        p *= trigram_prob(
            tokens[i - 2], tokens[i - 1], tokens[i]
        )
    return p


print("\nTASK 6: TRIGRAM MODEL")
print("Trigram-model sentence probabilities:")
for s in test:
    print(repr(s), fmt(trigram_sentence_prob(s)))

show_table(
    "Distinct N-gram counts",
    [
        ("Unigrams", len(unigram_counts)),
        ("Bigrams", len(bigram_counts)),
        ("Trigrams", len(trigram_counts))
    ],
    ["N-gram type", "Distinct count"]
)


def unseen_ngrams(sentence, n, counts):
    tokens = tokenize(sentence)
    return [
        gram for gram in get_ngrams(tokens, n)
        if gram not in counts
    ]


print("\nUnseen bigrams and trigrams in test sentences:")
for s in test:
    ub = unseen_ngrams(s, 2, bigram_counts)
    ut = unseen_ngrams(s, 3, trigram_counts)

    print("\nSentence:", s)
    print("Unseen bigrams:", ub)
    print("Unseen trigrams:", ut)

    if len(ub) > len(ut):
        print("More unseen N-grams: bigram model")
    elif len(ut) > len(ub):
        print("More unseen N-grams: trigram model")
    else:
        print("Equal numbers of unseen N-grams")



TASK 6: TRIGRAM MODEL
Trigram-model sentence probabilities:
'the cat sat on the rug' 0.1250
'the fish sat on the mat' 0.0000

Distinct N-gram counts
N-gram type | Distinct count
-----------------------------------------------------------------
Unigrams | 12
Bigrams | 18
Trigrams | 19

Unseen bigrams and trigrams in test sentences:

Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Equal numbers of unseen N-grams

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
More unseen N-grams: trigram model


# **Task 7: Zero-probability problem**
1.	Write unseen_bigrams(sentence) that returns every bigram of the sentence that never occurs in training.
2.	Run it on both test sentences and print the bigram MLE probability of each. Confirm that one of them is 0.
3.	Count how many of the possible word pairs (words only, markers excluded) never occur in training, and print this as a percentage.
4.	Write safe_log(p) that returns negative infinity for p = 0 instead of raising a math error, and use it to compute the log probability of the second test sentence.


In [21]:
def unseen_bigrams(sentence):
    return unseen_ngrams(sentence, 2, bigram_counts)


print("\nTASK 7: ZERO-PROBABILITY PROBLEM")
for s in test:
    print("\nSentence:", s)
    print("Unseen bigrams:", unseen_bigrams(s))
    print("Bigram MLE probability:", fmt(bigram_sentence_prob(s)))

print(
    "\nAt least one test sentence has zero bigram probability:",
    any(bigram_sentence_prob(s) == 0 for s in test)
)

# Possible ordered pairs of lexical words only.
lexical_vocab = sorted(Vocab - {"</s>"})
possible_pairs = {
    (w1, w2)
    for w1 in lexical_vocab
    for w2 in lexical_vocab
}

observed_lexical_pairs = {
    (w1, w2)
    for (w1, w2), count in bigram_counts.items()
    if count > 0
    and w1 in lexical_vocab
    and w2 in lexical_vocab
}

missing_pairs = possible_pairs - observed_lexical_pairs
missing_percentage = 100 * len(missing_pairs) / len(possible_pairs)

print("Possible lexical word pairs:", len(possible_pairs))
print("Observed lexical word pairs:", len(observed_lexical_pairs))
print("Unseen lexical word pairs:", len(missing_pairs))
print("Percentage unseen:", fmt(missing_percentage) + "%")


def safe_log(p):
    return log(p) if p > 0 else -inf


second_test_probability = bigram_sentence_prob(test[1])
print(
    "Log probability of second test sentence:",
    fmt(safe_log(second_test_probability))
)




TASK 7: ZERO-PROBABILITY PROBLEM

Sentence: the cat sat on the rug
Unseen bigrams: []
Bigram MLE probability: 0.0156

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE probability: 0.0000

At least one test sentence has zero bigram probability: True
Possible lexical word pairs: 100
Observed lexical word pairs: 13
Unseen lexical word pairs: 87
Percentage unseen: 87.0000%
Log probability of second test sentence: -inf


# Task 8: Smoothing
1.	Write bigram_prob_addone(prev, w) using (C(v, w) + 1) / (C(v) + V).
2.	Print P(sat | fish) and P(cat | the) under both MLE and add-one smoothing, side by side.
3.	Loop over every possible previous word and assert that its smoothed next-word probabilities over all V words sum to 1.
4.	Write bigram_prob_addk(prev, w, k) and print P(sat | fish) for k = 1, 0.5, 0.1 and 0.01 in a table.


In [24]:
from collections import Counter

V = len(Vocab)

# Count all bigrams beginning with each previous word
bigram_context_counts = Counter()

for (prev, word), count in bigram_counts.items():
    bigram_context_counts[prev] += count


# Add-k smoothing
def bigram_prob_addk(prev, w, k):
    denominator = bigram_context_counts[prev] + k * V

    if denominator == 0:
        return 0.0

    return (bigram_counts[(prev, w)] + k) / denominator


def bigram_prob_addone(prev, w):
    return bigram_prob_addk(prev, w, 1)


# Compare MLE and add-one smoothing
smoothing_rows = [
    (
        "P(sat | fish)",
        f"{bigram_prob('fish', 'sat'):.4f}",
        f"{bigram_prob_addone('fish', 'sat'):.4f}"
    ),
    (
        "P(cat | the)",
        f"{bigram_prob('the', 'cat'):.4f}",
        f"{bigram_prob_addone('the', 'cat'):.4f}"
    )
]

show_table(
    "MLE versus add-one smoothing",
    smoothing_rows,
    ["Probability", "MLE", "Add-one"]
)


# Check distributions over all vocabulary words
print("\nChecking add-one distributions:")

for prev in Vocab:
    total = sum(
        bigram_prob_addone(prev, w)
        for w in Vocab
    )
    assert abs(total - 1.0) < 1e-12, (prev, total)

print("All smoothed distributions sum to 1.")


# Add-k probabilities for different k values
addk_rows = [
    (k, f"{bigram_prob_addk('fish', 'sat', k):.4f}")
    for k in [1, 0.5, 0.1, 0.01]
]

show_table(
    "Add-k: P(sat | fish)",
    addk_rows,
    ["k", "Probability"]
)


MLE versus add-one smoothing
Probability | MLE | Add-one
-----------------------------------------------------------------
P(sat | fish) | 0.0000 | 0.0833
P(cat | the) | 0.2500 | 0.1579

Checking add-one distributions:
All smoothed distributions sum to 1.

Add-k: P(sat | fish)
k | Probability
-----------------------------------------------------------------
1 | 0.0833
0.5 | 0.0769
0.1 | 0.0476
0.01 | 0.0090


# **Task 9:** Perplexity and model comparison
1.	Write perplexity(sentences, prob_fn) that returns infinity when any probability is 0.
2.	Compute the perplexity of the two test sentences together for the bigram MLE model, the bigram add-one model and the unigram add-one model, where P(w) = (C(w) + 1) / (N + V). Print one results table.
3.	Compute the perplexity of the add-k bigram model for k = 1, 0.5, 0.1 and 0.01. Print the table and the best k.
4.	Train only on the first three training sentences (build V from all four) and pick the best k using the fourth sentence as validation data. Report the test perplexity with that k, and compare it with the best k found directly on the test set.


In [25]:
def unigram_prob_addone(w):
    return (unigram_counts[(w,)] + 1) / (N + V)


def perplexity(sentences, prob_fn):
    """
    prob_fn(prev, word) returns the probability of a predicted
    token. Each sentence starts with <s>; the start marker itself
    is not predicted, but </s> is included.
    """
    log_probability_sum = 0.0
    predicted_tokens = 0

    for sentence in sentences:
        tokens = tokenize(sentence)
        for i in range(1, len(tokens)):
            p = prob_fn(tokens[i - 1], tokens[i])
            if p == 0:
                return inf
            log_probability_sum += log(p)
            predicted_tokens += 1

    if predicted_tokens == 0:
        return 1.0

    return exp(-log_probability_sum / predicted_tokens)


def unigram_addone_scorer(prev, w):
    return unigram_prob_addone(w)


def bigram_addone_scorer(prev, w):
    return bigram_prob_addone(prev, w)


def make_addk_scorer(counts, k, vocab_size):
    """Create a bigram scorer for a specified training Counter."""
    def scorer(prev, w):
        denominator = counts[(prev,)] + k * vocab_size
        if denominator == 0:
            return 0.0
        return (counts[(prev, w)] + k) / denominator
    return scorer


test_rows = [
    (
        "Bigram MLE",
        fmt(perplexity(test, bigram_prob))
    ),
    (
        "Bigram add-one",
        fmt(perplexity(test, bigram_addone_scorer))
    ),
    (
        "Unigram add-one",
        fmt(perplexity(test, unigram_addone_scorer))
    )
]

show_table(
    "Perplexity on both test sentences",
    test_rows,
    ["Model", "Perplexity"]
)

k_values = [1, 0.5, 0.1, 0.01]
test_addk_results = {
    k: perplexity(test, make_addk_scorer(bigram_counts, k, V))
    for k in k_values
}

show_table(
    "Add-k bigram perplexity on test set",
    [(k, fmt(p)) for k, p in test_addk_results.items()],
    ["k", "Perplexity"]
)

best_test_k = min(test_addk_results, key=test_addk_results.get)
print("Best k selected directly on test set:", best_test_k)


Perplexity on both test sentences
Model | Perplexity
-----------------------------------------------------------------
Bigram MLE | inf
Bigram add-one | 5.7969
Unigram add-one | 10.1247

Add-k bigram perplexity on test set
k | Perplexity
-----------------------------------------------------------------
1 | 4.3866
0.5 | 2.8246
0.1 | 0.7807
0.01 | 0.0973
Best k selected directly on test set: 0.01


# Validation: train on the first three sentences only.
# Keep V built from all four training sentences.

In [26]:
validation_train_tokens = train_tokens[:3]

validation_unigrams = Counter()
validation_bigrams = Counter()

for tokens in validation_train_tokens:
    validation_unigrams.update(get_ngrams(tokens, 1))
    validation_bigrams.update(get_ngrams(tokens, 2))


def make_validation_addk_scorer(k):
    def scorer(prev, w):
        denominator = validation_unigrams[(prev,)] + k * V
        if denominator == 0:
            return 0.0
        return (
            validation_bigrams[(prev, w)] + k
        ) / denominator
    return scorer


validation_sentence = [train[3]]

validation_results = {
    k: perplexity(validation_sentence, make_validation_addk_scorer(k))
    for k in k_values
}

show_table(
    "Validation perplexity (fourth training sentence)",
    [(k, fmt(p)) for k, p in validation_results.items()],
    ["k", "Validation perplexity"]
)

best_validation_k = min(
    validation_results, key=validation_results.get
)

print("Best k selected on validation data:", best_validation_k)

# Evaluate on the two test sentences using the first-three-sentence
# training counts and the validation-selected k.
validation_selected_scorer = make_validation_addk_scorer(
    best_validation_k
)

validation_selected_test_perplexity = perplexity(
    test, validation_selected_scorer
)

print(
    "Test perplexity using validation-selected k:",
    fmt(validation_selected_test_perplexity)
)
print(
    "Test perplexity using test-selected k:",
    fmt(test_addk_results[best_test_k])
)
print(
    "Note: the latter is a test-set-selected score, not an "
    "unbiased final evaluation."
)


Validation perplexity (fourth training sentence)
k | Validation perplexity
-----------------------------------------------------------------
1 | 8.5857
0.5 | 8.0163
0.1 | 8.0260
0.01 | 13.1304
Best k selected on validation data: 0.5
Test perplexity using validation-selected k: 4.4539
Test perplexity using test-selected k: 0.0973
Note: the latter is a test-set-selected score, not an unbiased final evaluation.
